In [ ]:
# STEP 1: Libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# STEP 2: Logistic Regression (Baseline)
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

print("Loading MNIST dataset...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X = X / 255.0  # normalize pixels

# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42)

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, solver='saga', multi_class='multinomial', n_jobs=-1)
clf.fit(X_train, y_train)

baseline_acc = clf.score(X_test, y_test)
print("Baseline Logistic Regression Accuracy:", baseline_acc)

# Confusion matrix
y_pred_baseline = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

print(classification_report(y_test, y_pred_baseline))

# STEP 3: CNN with TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# load data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# preprocess
x_train = x_train.reshape(-1,28,28,1).astype('float32')/255.0
x_test  = x_test.reshape(-1,28,28,1).astype('float32')/255.0
y_train = to_categorical(y_train,10)
y_test  = to_categorical(y_test,10)

# build model
model = models.Sequential([
    layers.Conv2D(32,(3,3),activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# train
print("Training CNN model...")
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=12,
    batch_size=128,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

# evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("CNN Test Accuracy:", test_acc)

# plot training history
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss")
plt.show()

# Confusion matrix for CNN
y_pred = model.predict(x_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)
cm_cnn = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm_cnn, annot=False, cmap="Greens")
plt.title("Confusion Matrix - CNN")
plt.show()

print(classification_report(y_true, y_pred))

# save model
model.save("mnist_cnn.h5")
print("Model saved as mnist_cnn.h5")